# CGRS on CIFAR-10 — All Phases
**Curvature-Guided Rank Selection (CGRS) for ViT-Base/16**  
Dataset: CIFAR-10 (10 classes)  
Target hardware: USC CARC V100 GPU  

### Phases
- **Phase 1** — Fixed-rank baselines (frozen backbone, full fine-tune, LoRA sweep r3–r64)
- **Phase 2** — Per-layer diagonal Fisher (λ_max) profiling → calibrate τ thresholds
- **Phase 3** — Global CGRS (bidirectional rank adaptation, shared τ)
- **Phase 4** — Per-Layer CGRS (independent rank per transformer block, C1/C2/C3)
- **Final** — Cross-phase comparison table + publication plots


In [1]:
# ----------------------------------------------------------------
# CELL 1 — Imports and Reproducibility
# ----------------------------------------------------------------
import os, json, random, time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model
from scipy.stats import pearsonr, spearmanr

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")


/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: Tesla V100-PCIE-32GB
VRAM: 34.1 GB


In [2]:
# ----------------------------------------------------------------
# CELL 2 — Config (CIFAR-10 specific)
# ----------------------------------------------------------------
LORA_RANKS = [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]

CONFIG = {
    # Model
    "model_name": "google/vit-base-patch16-224",
    "num_classes": 10,          # CIFAR-10
    # Data
    "train_size": 45000,
    # Training
    "batch_size": 16,
    "epochs": 3,
    "weight_decay": 0.01,
    # Learning rates
    "lr_full": 5e-5,
    "lr_frozen": 1e-3,
    "lr_lora": 5e-4,
    # LoRA
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "target_modules": ["query", "value"],
    "lora_ranks": LORA_RANKS,
}

# ── Directories ──────────────────────────────────────────────────────────────
PROJECT_BASE = os.path.expanduser("~/CGRS_CIFAR10")
PHASE1_DIR = os.path.join(PROJECT_BASE, "phase1_results")
PHASE2_DIR = os.path.join(PROJECT_BASE, "phase2_results")
PHASE3_DIR = os.path.join(PROJECT_BASE, "phase3_results")
PHASE4_DIR = os.path.join(PROJECT_BASE, "phase4_results")
PLOTS_DIR  = os.path.join(PROJECT_BASE, "plots")
DATA_DIR   = os.path.join(PROJECT_BASE, "data")
for d in [PHASE1_DIR, PHASE2_DIR, PHASE3_DIR, PHASE4_DIR, PLOTS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

with open(os.path.join(PHASE1_DIR, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Config saved.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


Config saved.
  model_name: google/vit-base-patch16-224
  num_classes: 10
  train_size: 45000
  batch_size: 16
  epochs: 3
  weight_decay: 0.01
  lr_full: 5e-05
  lr_frozen: 0.001
  lr_lora: 0.0005
  lora_alpha: 16
  lora_dropout: 0.1
  target_modules: ['query', 'value']
  lora_ranks: [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]


In [3]:
# ----------------------------------------------------------------
# CELL 3 — Dataset and DataLoaders (CIFAR-10)
# ----------------------------------------------------------------
processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

print("Loading CIFAR-10...")
full_train = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=transform)
test_dataset= datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

val_size = len(full_train) - CONFIG["train_size"]
train_dataset, val_dataset = random_split(
    full_train, [CONFIG["train_size"], val_size],
    generator=torch.Generator().manual_seed(SEED)
)

NUM_WORKERS = 4
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
# Small-batch probe loader for Fisher (batchsize=4)
probe_loader = DataLoader(train_dataset, batch_size=4, shuffle=False,
                          num_workers=0, pin_memory=False)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")
print(f"Probe loader: {len(probe_loader)} batches (batchsize=4)")


/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading CIFAR-10...
Files already downloaded and verified
Files already downloaded and verified
Train: 45,000
Val:   5,000
Test:  10,000
Probe loader: 11250 batches (batchsize=4)


In [4]:
# ----------------------------------------------------------------
# CELL 4 — Model Builders and Training Helpers
# ----------------------------------------------------------------
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total

def build_full_model():
    model = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"], num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True)
    t, total = count_parameters(model)
    print(f"Full model — Trainable: {t:,} / {total:,} ({100*t/total:.2f}%)")
    return model.to(device)

def build_frozen_model():
    model = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"], num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    t, total = count_parameters(model)
    print(f"Frozen model — Trainable: {t:,} / {total:,} ({100*t/total:.4f}%)")
    return model.to(device)

def build_lora_model_verbose(r):
    base = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"], num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True)
    cfg = LoraConfig(r=r, lora_alpha=CONFIG["lora_alpha"],
                     lora_dropout=CONFIG["lora_dropout"],
                     target_modules=CONFIG["target_modules"], bias="none")
    model = get_peft_model(base, cfg).to(device)
    t, total = count_parameters(model)
    print(f"LoRA r={r:4d} — Trainable: {t:10,} / {total:,} ({100*t/total:.3f}%)")
    return model, t

def build_lora_model_silent(r):
    base = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"], num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True)
    cfg = LoraConfig(r=r, lora_alpha=CONFIG["lora_alpha"],
                     lora_dropout=CONFIG["lora_dropout"],
                     target_modules=CONFIG["target_modules"], bias="none")
    return get_peft_model(base, cfg).to(device)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        out = model(images)
        loss_sum += criterion(out.logits, labels).item()
        correct  += (out.logits.argmax(dim=1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total

def train_model(model, lr, label):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CONFIG["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["epochs"] * len(train_loader))
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for step, (images, labels) in enumerate(train_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), 1.0)
            optimizer.step(); scheduler.step()
            running_loss += loss.item()
            correct      += (out.logits.argmax(1) == labels).sum().item()
            total        += labels.size(0)
            if (step + 1) % 500 == 0:
                print(f"  {label} Epoch {epoch} Step {step+1}/{len(train_loader)} "
                      f"Loss {running_loss/(step+1):.4f}")
        val_loss, val_acc = evaluate(model, val_loader)
        print(f"  {label} Epoch {epoch}/{CONFIG['epochs']} "
              f"Train Loss {running_loss/len(train_loader):.4f} "
              f"Train Acc {100*correct/total:.2f}% "
              f"Val Loss {val_loss:.4f} Val Acc {val_acc:.2f}%")
    return model

print("All helpers ready.")


All helpers ready.


## Phase 1 — Fixed-Rank Baselines

In [5]:
# ----------------------------------------------------------------
# CELL 5 — Load Phase 1 results if they exist (resume point)
# ----------------------------------------------------------------
results_path = os.path.join(PHASE1_DIR, "results.json")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("Loaded existing results.json:")
    for k, v in results.items():
        print(f"  {k:28s} Test Acc {v['test_acc']:.2f}")
else:
    results = {}
    print("Starting fresh results dict.")


Loaded existing results.json:
  Full fine-tune               Test Acc 98.81
  Frozen backbone              Test Acc 96.95
  LoRA r3                      Test Acc 98.52
  LoRA r5                      Test Acc 98.69
  LoRA r6                      Test Acc 98.64
  LoRA r10                     Test Acc 98.65
  LoRA r12                     Test Acc 98.70
  LoRA r16                     Test Acc 98.74
  LoRA r30                     Test Acc 98.74
  LoRA r52                     Test Acc 98.83
  LoRA r64                     Test Acc 98.72
  LoRA r80                     Test Acc 98.82
  LoRA r100                    Test Acc 98.74
  LoRA r128                    Test Acc 98.70
  LoRA r150                    Test Acc 98.71
  LoRA r200                    Test Acc 98.68


In [6]:
# ----------------------------------------------------------------
# CELL 6 — Experiment A: Full Fine-Tuning (upper bound)
# ----------------------------------------------------------------
print("=" * 60)
print("EXPERIMENT A — Full Fine-Tuning")
print("=" * 60)
if "Full fine-tune" not in results:
    modelA = build_full_model()
    modelA = train_model(modelA, lr=CONFIG["lr_full"], label="Full")
    loss_A, acc_A = evaluate(modelA, test_loader)
    t_A, _ = count_parameters(modelA)
    print(f"  Result — Test Loss {loss_A:.4f}  Test Acc {acc_A:.2f}%")
    results["Full fine-tune"] = {"test_loss": loss_A, "test_acc": acc_A,
                                  "trainable_params": t_A}
    with open(results_path, "w") as f: json.dump(results, f, indent=2)
    del modelA; torch.cuda.empty_cache()
else:
    print(f"  Already done: {results['Full fine-tune']['test_acc']:.2f}% — skipping.")


EXPERIMENT A — Full Fine-Tuning
  Already done: 98.81% — skipping.


In [7]:
# ----------------------------------------------------------------
# CELL 7 — Experiment B: Frozen Backbone (lower bound)
# ----------------------------------------------------------------
print("=" * 60)
print("EXPERIMENT B — Frozen Backbone")
print("=" * 60)
if "Frozen backbone" not in results:
    modelB = build_frozen_model()
    t_B, _ = count_parameters(modelB)
    modelB = train_model(modelB, lr=CONFIG["lr_frozen"], label="Frozen")
    loss_B, acc_B = evaluate(modelB, test_loader)
    print(f"  Result — Test Loss {loss_B:.4f}  Test Acc {acc_B:.2f}%")
    results["Frozen backbone"] = {"test_loss": loss_B, "test_acc": acc_B,
                                   "trainable_params": t_B}
    with open(results_path, "w") as f: json.dump(results, f, indent=2)
    del modelB; torch.cuda.empty_cache()
else:
    print(f"  Already done: {results['Frozen backbone']['test_acc']:.2f}% — skipping.")


EXPERIMENT B — Frozen Backbone
  Already done: 96.95% — skipping.


In [8]:
# ----------------------------------------------------------------
# CELL 8 — Experiment C: LoRA Sweep (resume-safe)
# ----------------------------------------------------------------
print("=" * 60)
print("EXPERIMENT C — LoRA Fixed-Rank Sweep")
print("=" * 60)
print(f"  Ranks: {LORA_RANKS}")
print(f"  Est. disk: {len(LORA_RANKS)*0.35:.1f} GB for .pt checkpoints")

for r in LORA_RANKS:
    key     = f"LoRA r{r}"
    ckpt_path = os.path.join(PHASE1_DIR, f"lora_r{r}_full.pt")
    if key in results and os.path.exists(ckpt_path):
        print(f"  r{r:4d} already done acc={results[key]['test_acc']:.2f}% — skipping.")
        continue
    print(f"\n{'='*55}")
    print(f"  Training LoRA r={r}  ({LORA_RANKS.index(r)+1}/{len(LORA_RANKS)})")
    print(f"{'='*55}")
    torch.cuda.empty_cache()
    modelC, t_C = build_lora_model_verbose(r)
    modelC = train_model(modelC, lr=CONFIG["lr_lora"], label=f"LoRA r{r}")
    loss_C, acc_C = evaluate(modelC, test_loader)
    print(f"  r{r} — Test Loss {loss_C:.4f}  Test Acc {acc_C:.2f}%")
    # Save full state dict (Phase 2 needs it)
    torch.save(modelC.state_dict(), ckpt_path)
    print(f"  Checkpoint saved: {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.0f} MB)")
    results[key] = {"test_loss": loss_C, "test_acc": acc_C,
                    "trainable_params": t_C, "ckpt_path": ckpt_path}
    with open(results_path, "w") as f: json.dump(results, f, indent=2)
    del modelC; torch.cuda.empty_cache()

print("\nAll LoRA ranks complete.")


EXPERIMENT C — LoRA Fixed-Rank Sweep
  Ranks: [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]
  Est. disk: 4.9 GB for .pt checkpoints
  r   3 already done acc=98.52% — skipping.
  r   5 already done acc=98.69% — skipping.
  r   6 already done acc=98.64% — skipping.
  r  10 already done acc=98.65% — skipping.
  r  12 already done acc=98.70% — skipping.
  r  16 already done acc=98.74% — skipping.
  r  30 already done acc=98.74% — skipping.
  r  52 already done acc=98.83% — skipping.
  r  64 already done acc=98.72% — skipping.
  r  80 already done acc=98.82% — skipping.
  r 100 already done acc=98.74% — skipping.
  r 128 already done acc=98.70% — skipping.
  r 150 already done acc=98.71% — skipping.
  r 200 already done acc=98.68% — skipping.

All LoRA ranks complete.


In [9]:
# ----------------------------------------------------------------
# CELL 9 — Phase 1 Summary Table
# ----------------------------------------------------------------
with open(results_path) as f: results = json.load(f)

TOTAL_PARAMS = 85_875_556
print("=" * 80)
print("PHASE 1 COMPLETE — CIFAR-10 Results")
print("=" * 80)
print(f"{'Method':22s}  {'Trainable Params':18s}  {'% Total':8s}  {'Test Loss':10s}  {'Test Acc':10s}")
print("-" * 80)
for method in ["Full fine-tune", "Frozen backbone"] + [f"LoRA r{r}" for r in LORA_RANKS]:
    if method not in results: continue
    v = results[method]
    t = v.get("trainable_params", 0)
    print(f"{method:22s}  {t:18,}  {100*t/TOTAL_PARAMS:8.3f}%  {v['test_loss']:10.4f}  {v['test_acc']:8.2f}%")
print("=" * 80)
print("Phase 1 complete. Proceed to Phase 2.")


PHASE 1 COMPLETE — CIFAR-10 Results
Method                  Trainable Params    % Total   Test Loss   Test Acc  
--------------------------------------------------------------------------------
Full fine-tune                  85,806,346    99.919%      0.0572     98.81%
Frozen backbone                      7,690     0.009%      0.1013     96.95%
LoRA r3                            110,592     0.129%      0.0616     98.52%
LoRA r5                            184,320     0.215%      0.0508     98.69%
LoRA r6                            221,184     0.258%      0.0496     98.64%
LoRA r10                           368,640     0.429%      0.0504     98.65%
LoRA r12                           442,368     0.515%      0.0506     98.70%
LoRA r16                           589,824     0.687%      0.0473     98.74%
LoRA r30                         1,105,920     1.288%      0.0449     98.74%
LoRA r52                         1,916,928     2.232%      0.0423     98.83%
LoRA r64                         2,3

## Phase 2 — Per-Layer Diagonal Fisher Profiling

In [10]:
# ----------------------------------------------------------------
# CELL 10 — Fisher Computation Functions
# ----------------------------------------------------------------
criterion_probe = nn.CrossEntropyLoss()
N_CURV_BATCHES = 64   # 64 × 4 = 256 samples — stable estimate

def compute_global_fisher(model, loader, n_batches=64):
    """Global diagonal Fisher: F_diag = (1/N) sum_i (grad_i loss)^2"""
    model.eval()
    fisher = None
    done = 0
    for images, labels in loader:
        if done >= n_batches: break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        model.zero_grad()
        out  = model(images)
        loss = criterion_probe(out.logits, labels)
        loss.backward()
        with torch.no_grad():
            grads = torch.cat([p.grad.detach().pow(2).flatten()
                               for p in model.parameters()
                               if p.requires_grad and p.grad is not None])
        fisher = grads if fisher is None else fisher + grads
        done += 1
    fisher /= done
    n = int(fisher.shape[0])
    lmax  = float(fisher.max().item())
    trace = float(fisher.sum().item())
    tnorm = trace / n
    conc  = lmax / (tnorm + 1e-12)
    return {"lambda_max": lmax, "trace": trace,
            "trace_normalized": tnorm, "concentration_ratio": conc, "num_params": n}

def compute_perlayer_fisher(model, loader, n_batches=64):
    """Per-layer diagonal Fisher using PEFT named parameters (guaranteed non-zero gradients)."""
    model.eval()
    accum = {}   # key: (layer_idx, proj) -> accumulated grad^2 tensor
    done = 0
    for images, labels in loader:
        if done >= n_batches: break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        model.zero_grad()
        out  = model(images)
        loss = criterion_probe(out.logits, labels)
        loss.backward()
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None or not param.requires_grad: continue
                if "lora_A" not in name and "lora_B" not in name: continue
                parts = name.split(".")
                layer_idx, proj = None, None
                for j, p in enumerate(parts):
                    if p == "layer" and j + 1 < len(parts):
                        try: layer_idx = int(parts[j+1])
                        except ValueError: pass
                    if p in ("query", "value"): proj = p
                if layer_idx is None or proj is None: continue
                key = (layer_idx, proj)
                g2  = param.grad.detach().pow(2).flatten()
                accum[key] = accum.get(key, torch.zeros_like(g2)) + g2
        done += 1
    model.zero_grad()
    result = {}
    for key, fisher in accum.items():
        fisher /= done
        result[key] = float(fisher.max().item())
    # fill missing
    for i in range(12):
        for proj in ("query", "value"):
            result.setdefault((i, proj), 0.0)
    return result

print(f"Fisher functions ready — {N_CURV_BATCHES} batches × 4 = {N_CURV_BATCHES*4} samples/rank")


Fisher functions ready — 64 batches × 4 = 256 samples/rank


In [11]:
# ----------------------------------------------------------------
# CELL 11 — Run Phase 2 Curvature Analysis (resume-safe)
# ----------------------------------------------------------------
curv_path = os.path.join(PHASE2_DIR, "curvature_results.json")
if os.path.exists(curv_path):
    with open(curv_path) as f: curvature_results = json.load(f)
    print(f"Loaded existing curvature_results.json ({len(curvature_results)} entries)")
else:
    curvature_results = {}

for r in LORA_RANKS:
    key      = f"r{r}"
    ckpt_path = os.path.join(PHASE1_DIR, f"lora_r{r}_full.pt")
    if key in curvature_results:
        print(f"  r{r:4d} curvature already computed — skipping.")
        continue
    if not os.path.exists(ckpt_path):
        print(f"  r{r:4d} checkpoint not found — run Phase 1 first.")
        continue
    print(f"\n{'='*55}")
    print(f"  Curvature LoRA r={r}  ({LORA_RANKS.index(r)+1}/{len(LORA_RANKS)})")
    print(f"{'='*55}")
    torch.cuda.empty_cache()
    model = build_lora_model_silent(r)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    del state; torch.cuda.empty_cache()
    print(f"  Weights loaded.")
    print(f"  Computing global Fisher ({N_CURV_BATCHES} batches × 4)...")
    gf = compute_global_fisher(model, probe_loader, n_batches=N_CURV_BATCHES)
    print(f"  lambda_max={gf['lambda_max']:.6f}  trace={gf['trace']:.4f}  "
          f"trace_norm={gf['trace_normalized']:.8f}  conc={gf['concentration_ratio']:.2f}")
    print(f"  Computing per-layer Fisher...")
    lf = compute_perlayer_fisher(model, probe_loader, n_batches=N_CURV_BATCHES)
    # serialise tuple keys
    lf_serial = {f"{k[0]}_{k[1]}": v for k, v in lf.items()}
    curvature_results[key] = {**gf, "per_layer": lf_serial}
    with open(curv_path, "w") as f: json.dump(curvature_results, f, indent=2)
    print(f"  Saved → curvature_results.json")
    del model; torch.cuda.empty_cache()

print("\nAll curvature analysis complete.")


Loaded existing curvature_results.json (14 entries)
  r   3 curvature already computed — skipping.
  r   5 curvature already computed — skipping.
  r   6 curvature already computed — skipping.
  r  10 curvature already computed — skipping.
  r  12 curvature already computed — skipping.
  r  16 curvature already computed — skipping.
  r  30 curvature already computed — skipping.
  r  52 curvature already computed — skipping.
  r  64 curvature already computed — skipping.
  r  80 curvature already computed — skipping.
  r 100 curvature already computed — skipping.
  r 128 curvature already computed — skipping.
  r 150 curvature already computed — skipping.
  r 200 curvature already computed — skipping.

All curvature analysis complete.


In [12]:
# ----------------------------------------------------------------
# CELL 12 — Phase 2 Summary + Calibrate τ Thresholds
# ----------------------------------------------------------------
with open(curv_path) as f: curvature_results = json.load(f)

print("=" * 75)
print("PHASE 2 COMPLETE — CIFAR-10 Curvature Results")
print("=" * 75)
print(f"{'Rank':8s}  {'lambda_max':14s}  {'trace':12s}  {'trace_norm':14s}  {'conc_ratio':12s}")
print("-" * 75)
for r in LORA_RANKS:
    key = f"r{r}"
    if key not in curvature_results: continue
    v = curvature_results[key]
    print(f"r{r:6d}  {v['lambda_max']:14.6f}  {v['trace']:12.4f}  "
          f"{v['trace_normalized']:14.8f}  {v['concentration_ratio']:12.2f}")
print("=" * 75)

# ── Calibrate τ from r16 per-layer λ_max ──────────────────────────────────
if "r16" in curvature_results:
    pl16 = curvature_results["r16"]["per_layer"]
    vals16 = list(pl16.values())
    TAU_C1 = float(np.percentile(vals16, 75))   # 75th percentile — conservative
    TAU_C2 = float(np.percentile(vals16, 50))   # 50th percentile — moderate
    TAU_PER_LAYER = {k: v for k, v in pl16.items()}   # per-layer own r16 λ_max
    TAU_C3_HALF   = {k: v * 0.5 for k, v in pl16.items()}  # corrected C3 (0.5× λ_max)
    print(f"\nτ Calibration from r16 per-layer λ_max ({len(vals16)} layer-proj pairs):")
    print(f"  Min:   {min(vals16):.6f}")
    print(f"  25th:  {float(np.percentile(vals16,25)):.6f}")
    print(f"  50th (C2 τ): {TAU_C2:.6f}")
    print(f"  75th (C1 τ): {TAU_C1:.6f}")
    print(f"  Max:   {max(vals16):.6f}")
    print(f"\n  TAU_C1 (75th pct, shared) = {TAU_C1:.6f}")
    print(f"  TAU_C2 (50th pct, shared) = {TAU_C2:.6f}")
    print(f"  TAU_C3 (per-layer own λ_max) — see dict TAU_PER_LAYER")
    print(f"  TAU_C3_HALF (per-layer 0.5× λ_max) — corrected C3")
else:
    print("WARNING: r16 not found in curvature_results. Run Phase 2 Cell 11 first.")
    TAU_C1, TAU_C2 = 0.001, 0.0005   # fallback
    TAU_PER_LAYER = {}; TAU_C3_HALF = {}


PHASE 2 COMPLETE — CIFAR-10 Curvature Results
Rank      lambda_max      trace         trace_norm      conc_ratio  
---------------------------------------------------------------------------
r     3        0.000036        0.0286      0.00000026        140.23
r     5        0.062021       14.2771      0.00007746        800.70
r     6        0.000078        0.0282      0.00000013        615.68
r    10        0.001887        0.8838      0.00000240        787.22
r    12        0.001110        0.4850      0.00000110       1012.65
r    16        0.000138        0.0920      0.00000016        884.02
r    30        0.000596        0.3450      0.00000031       1910.56
r    52        0.000007        0.0141      0.00000001       1021.21
r    64        0.000006        0.0109      0.00000000       1218.94
r    80        0.000001        0.0027      0.00000000       1429.15
r   100        0.000003        0.0122      0.00000000        946.58
r   128        0.000001        0.0050      0.00000000       1

## Phase 3 — Global CGRS (bidirectional, shared τ)

In [13]:
# ----------------------------------------------------------------
# CELL 13 — SVD Rank Transition + Global Fisher Probe
# ----------------------------------------------------------------
LORA_RANK_LADDER = [3, 5, 6, 10, 12, 16, 30, 52, 64]

def transition_lora_rank(old_model, old_r, new_r):
    """SVD-based weight transfer when rank changes. Reads actual tensor shapes (bug-fixed)."""
    old_params = {n: p.data.clone().cpu() for n, p in old_model.named_parameters()}
    new_model  = build_lora_model_silent(new_r); new_model.cpu()
    new_param_dict = dict(new_model.named_parameters())
    # Copy non-LoRA weights
    for name, old_data in old_params.items():
        if "lora_A" not in name and "lora_B" not in name:
            if name in new_param_dict:
                new_param_dict[name].data.copy_(old_data)
    # SVD-transfer LoRA weights
    lora_bases = {}
    for name, data in old_params.items():
        if "lora_A" in name:
            base = name[:name.index("lora_A")]
            lora_bases.setdefault(base, {})["A"] = data
            lora_bases[base]["A_name"] = name
        elif "lora_B" in name:
            base = name[:name.index("lora_B")]
            lora_bases.setdefault(base, {})["B"] = data
            lora_bases[base]["B_name"] = name
    for base, pair in lora_bases.items():
        if "A" not in pair or "B" not in pair: continue
        A, B = pair["A"].float(), pair["B"].float()
        actual_old_r = A.shape[0]   # read from tensor shape (bug-fix)
        if new_r > actual_old_r:
            new_A = torch.zeros(new_r, A.shape[1]); new_A[:actual_old_r] = A
            new_B = torch.zeros(B.shape[0], new_r); new_B[:, :actual_old_r] = B
        else:
            W = B @ A
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                sqrt_S = torch.sqrt(S[:new_r].clamp(min=0.0))
                new_B = U[:, :new_r] * sqrt_S
                new_A = Vh[:new_r] * sqrt_S.unsqueeze(1)
            except RuntimeError:
                new_A = A[:new_r]; new_B = B[:, :new_r]
        for name, param in new_model.named_parameters():
            if name.startswith(base) and "lora_A" in name:
                param.data.copy_(new_A.to(param.dtype))
            elif name.startswith(base) and "lora_B" in name:
                param.data.copy_(new_B.to(param.dtype))
    return new_model.to(device)

def probe_global_fisher(model, loader, n_batches):
    """Global λ_max for CGRS rank decision."""
    was_training = model.training
    model.eval()
    fisher = None; done = 0
    for images, labels in loader:
        if done >= n_batches: break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        model.zero_grad()
        out  = model(images)
        loss = criterion_probe(out.logits, labels)
        loss.backward()
        with torch.no_grad():
            grads = torch.cat([p.grad.detach().pow(2).flatten()
                               for p in model.parameters()
                               if p.requires_grad and p.grad is not None])
        fisher = grads if fisher is None else fisher + grads
        done += 1
    model.zero_grad()
    if was_training: model.train()
    if fisher is None or done == 0: return float("inf")
    return float((fisher / done).max().item())

print("transition_lora_rank and probe_global_fisher ready.")


transition_lora_rank and probe_global_fisher ready.


In [14]:
# ----------------------------------------------------------------
# CELL 14 — Global CGRS Training Loop (bidirectional, resume-safe)
# ----------------------------------------------------------------
P3 = {
    "r_init":       16,
    "r_min":         3,
    "r_max":        64,
    "rank_list":    [r for r in LORA_RANK_LADDER if r <= 64],
    "check_every":  200,
    "cooldown":     100,
    "probe_batches": 8,
    "epochs":       CONFIG["epochs"],
}

p3_json_path = os.path.join(PHASE3_DIR, "phase3_results.json")
CKPT_BASE_P3 = os.path.join(PHASE3_DIR, "checkpoints")
os.makedirs(CKPT_BASE_P3, exist_ok=True)

if os.path.exists(p3_json_path):
    with open(p3_json_path) as f: phase3_results = json.load(f)
else:
    phase3_results = {}

def run_cgrs_global(tau, label):
    RANKLIST = P3["rank_list"]; K = P3["check_every"]
    COOLDOWN = P3["cooldown"];  NPROBE = P3["probe_batches"]
    EPOCHS   = P3["epochs"];    r_init = P3["r_init"]
    ckpt_dir = os.path.join(CKPT_BASE_P3, label)
    os.makedirs(ckpt_dir, exist_ok=True)
    state_path = os.path.join(ckpt_dir, "state.json")
    total_steps = EPOCHS * len(train_loader)
    print("=" * 65)
    print(f"  {label}  tau={tau:.4f}  r_init={r_init}")
    print(f"  check_every={K}  cooldown={COOLDOWN}  probe={NPROBE}")
    print("=" * 65)
    # Resume
    if os.path.exists(state_path):
        with open(state_path) as f: state = json.load(f)
        start_epoch = state["start_epoch"]; start_batch = state["start_batch"]
        current_r   = state["current_r"];   global_step = state["global_step"]
        steps_since = state["steps_since_chg"]
        rank_traj   = state["rank_trajectory"]
        lambda_traj = state["lambda_trajectory"]
        rank_changes= state["rank_changes"]
        rank_step_count = defaultdict(int, {int(k): v for k, v in state["rank_step_count"].items()})
        model = build_lora_model_silent(current_r)
        model.load_state_dict(torch.load(os.path.join(ckpt_dir, "model.pt"), map_location=device))
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
        optimizer.load_state_dict(torch.load(os.path.join(ckpt_dir, "optimizer.pt"), map_location=device))
        scheduler.load_state_dict(torch.load(os.path.join(ckpt_dir, "scheduler.pt"), map_location=device))
        print(f"  RESUME epoch={start_epoch} batch={start_batch} r={current_r} step={global_step}")
    else:
        start_epoch = 1; start_batch = 0; current_r = r_init; global_step = 0
        steps_since = COOLDOWN; rank_traj = []; lambda_traj = []; rank_changes = []
        rank_step_count = defaultdict(int)
        torch.cuda.empty_cache()
        model = build_lora_model_silent(r_init)
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
        print(f"  FRESH  r={r_init}")
    criterion = nn.CrossEntropyLoss()
    def save_ckpt(ep, bi, rl=0.0, cr=0, tt=0):
        torch.save(model.state_dict(), os.path.join(ckpt_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
        torch.save(scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
        with open(state_path, "w") as f:
            json.dump({"start_epoch": ep, "start_batch": bi, "current_r": current_r,
                       "global_step": global_step, "steps_since_chg": steps_since,
                       "rank_trajectory": rank_traj, "lambda_trajectory": lambda_traj,
                       "rank_changes": rank_changes,
                       "rank_step_count": dict(rank_step_count),
                       "running_loss": rl, "correct": cr, "total": tt}, f, indent=2)
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for batch_idx, (images, labels) in enumerate(train_loader):
            if epoch == start_epoch and batch_idx < start_batch: continue
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), 1.0)
            optimizer.step(); scheduler.step()
            running_loss += loss.item(); global_step += 1; steps_since += 1
            correct += (out.logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
            rank_step_count[current_r] += 1
            # ── Rank check ──────────────────────────────────────────────────
            if steps_since >= K:
                lmax = probe_global_fisher(model, probe_loader, NPROBE)
                rank_traj.append([global_step, current_r])
                lambda_traj.append([global_step, lmax])
                r_idx = RANKLIST.index(current_r) if current_r in RANKLIST                         else min(range(len(RANKLIST)), key=lambda i: abs(RANKLIST[i]-current_r))
                new_r = current_r; reason = "stable"
                if lmax > tau and r_idx < len(RANKLIST) - 1:
                    cand = RANKLIST[r_idx + 1]
                    if cand <= P3["r_max"]: new_r = cand; reason = f"UP  lmax={lmax:.5f} τ={tau:.5f}"
                elif lmax < tau and r_idx > 0:
                    cand = RANKLIST[r_idx - 1]
                    if cand >= P3["r_min"]: new_r = cand; reason = f"DOWN lmax={lmax:.5f} τ={tau:.5f}"
                if new_r != current_r:
                    cur_lr = float(scheduler.get_last_lr()[0])
                    print(f"  Step {global_step:5d} Rank {current_r:2d}→{new_r:2d}  {reason}")
                    model = transition_lora_rank(model, current_r, new_r)
                    current_r = new_r
                    remaining = max(total_steps - global_step, 1)
                    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                           lr=cur_lr, weight_decay=CONFIG["weight_decay"])
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining)
                    rank_changes.append({"step": global_step, "new_r": new_r, "lambda": lmax, "reason": reason})
                    torch.cuda.empty_cache()
                    save_ckpt(epoch, batch_idx+1)
                    print(f"  Checkpoint saved after rank change.")
                steps_since = 0
        val_loss, val_acc = evaluate(model, val_loader)
        train_acc = 100.0 * correct / total
        print(f"  {label} Epoch {epoch}/{EPOCHS} r={current_r} "
              f"Train Loss {running_loss/len(train_loader):.4f} Train Acc {train_acc:.2f}% "
              f"Val Loss {val_loss:.4f} Val Acc {val_acc:.2f}%")
        save_ckpt(epoch+1, 0)
        print(f"  Checkpoint saved after epoch {epoch}.")
        start_batch = 0
    test_loss, test_acc = evaluate(model, test_loader)
    total_done = max(sum(rank_step_count.values()), 1)
    avg_rank = sum(r * c for r, c in rank_step_count.items()) / total_done
    print(f"\n  {label} DONE  Test Acc={test_acc:.2f}%  Avg Rank={avg_rank:.2f}  "
          f"Changes={len(rank_changes)}")
    del model; torch.cuda.empty_cache()
    return {"label": label, "tau": tau, "r_init": r_init, "test_acc": test_acc,
            "test_loss": test_loss, "avg_rank": avg_rank, "final_r": current_r,
            "n_rank_changes": len(rank_changes), "rank_changes": rank_changes,
            "rank_trajectory": rank_traj, "lambda_trajectory": lambda_traj}

print("run_cgrs_global ready.")


run_cgrs_global ready.


In [15]:
# ----------------------------------------------------------------
# CELL 15 — Run Phase 3: Three τ values (resume-safe)
# ----------------------------------------------------------------
# Calibrate tau_high / tau_mid / tau_low from Phase 2
# Falls back to reasonable values if Phase 2 not yet run
if "r16" in curvature_results:
    lmax_vals = [curvature_results[f"r{r}"]["lambda_max"]
                 for r in LORA_RANKS if f"r{r}" in curvature_results]
    tau_high = float(np.percentile(lmax_vals, 75))
    tau_mid  = float(np.percentile(lmax_vals, 50))
    tau_low  = float(np.percentile(lmax_vals, 25))
else:
    tau_high, tau_mid, tau_low = 0.010, 0.005, 0.002  # fallback
    print("WARNING: Using fallback tau values. Run Phase 2 first for proper calibration.")

print(f"Phase 3 τ values:")
print(f"  tau_high (75th pct) = {tau_high:.6f}")
print(f"  tau_mid  (50th pct) = {tau_mid:.6f}")
print(f"  tau_low  (25th pct) = {tau_low:.6f}")

for tau, label in [
    (tau_high, f"CGRS_tau_high_{tau_high:.4f}"),
    (tau_mid,  f"CGRS_tau_mid_{tau_mid:.4f}"),
    (tau_low,  f"CGRS_tau_low_{tau_low:.4f}"),
]:
    if label in phase3_results and "test_acc" in phase3_results[label]:
        print(f"  {label} already done ({phase3_results[label]['test_acc']:.2f}%) — skipping.")
        continue
    result = run_cgrs_global(tau=tau, label=label)
    phase3_results[label] = result
    with open(p3_json_path, "w") as f: json.dump(phase3_results, f, indent=2)
    print(f"  Saved phase3_results.json")


Phase 3 τ values:
  tau_high (75th pct) = 0.000481
  tau_mid  (50th pct) = 0.000022
  tau_low  (25th pct) = 0.000002
  CGRS_tau_high_0.0005 already done (98.44%) — skipping.
  CGRS_tau_mid_0.0000 already done (98.16%) — skipping.
  CGRS_tau_low_0.0000 already done (97.39%) — skipping.


In [16]:
# ----------------------------------------------------------------
# CELL 16 — Phase 3 Summary
# ----------------------------------------------------------------
with open(p3_json_path) as f: phase3_results = json.load(f)

print("=" * 75)
print("PHASE 3 — Global CGRS Results (CIFAR-10)")
print("=" * 75)
print(f"{'Run':40s}  {'τ':10s}  {'Avg Rank':10s}  {'Test Acc':10s}  {'Changes':8s}")
print("-" * 75)
for k, v in phase3_results.items():
    if "test_acc" not in v: continue
    print(f"{k:40s}  {v['tau']:10.4f}  {v['avg_rank']:10.2f}  {v['test_acc']:8.2f}%  {v['n_rank_changes']:8d}")


PHASE 3 — Global CGRS Results (CIFAR-10)
Run                                       τ           Avg Rank    Test Acc    Changes 
---------------------------------------------------------------------------
CGRS_tau_high_0.0005                          0.0005        4.65     98.44%         9
CGRS_tau_mid_0.0000                           0.0000       21.03     98.16%        40
CGRS_tau_low_0.0000                           0.0000       31.95     97.39%        33


## Phase 4 — Per-Layer CGRS

In [17]:
# ----------------------------------------------------------------
# CELL 17 — Per-Layer CGRS Training Loop
# ----------------------------------------------------------------
P4 = {
    "r_init":       16,
    "r_min":         3,
    "r_max":        64,
    "rank_list":    [3, 5, 6, 10, 12, 16, 30, 52, 64],
    "check_every":  200,
    "cooldown":     100,
    "probe_batches": 8,
    "epochs":       CONFIG["epochs"],
}

p4_json_path = os.path.join(PHASE4_DIR, "phase4_results.json")
if os.path.exists(p4_json_path):
    with open(p4_json_path) as f: phase4_results = json.load(f)
else:
    phase4_results = {}

def run_perlayer_cgrs(run_name, tau_config, r_init=16, expand_only=True):
    """
    Per-layer CGRS.
    tau_config: float (shared) OR dict {"layer_proj_key": float} for per-layer τ.
    The model uses a single global PEFT rank; per-layer logic tracks which layer
    triggers expansion and uses the MOST EXPANSIVE layer's rank as the global rank.
    """
    is_per_layer = isinstance(tau_config, dict)
    RANKLIST = P4["rank_list"]; K = P4["check_every"]
    COOLDOWN = P4["cooldown"];  NPROBE = P4["probe_batches"]
    EPOCHS   = P4["epochs"]
    total_steps = EPOCHS * len(train_loader)
    ckpt_dir = os.path.join(PHASE4_DIR, run_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    state_path = os.path.join(ckpt_dir, "state.json")
    print("=" * 65)
    print(f"  {run_name}")
    print(f"  tau={'per-layer' if is_per_layer else f'{tau_config:.4e}'}  "
          f"r_init={r_init}  expand_only={expand_only}")
    print("=" * 65)
    layer_ranks     = {i: r_init for i in range(12)}
    cooldown_left   = {i: 0 for i in range(12)}
    rank_history    = {i: [[0, r_init]] for i in range(12)}
    lambda_history  = {i: [] for i in range(12)}
    rank_changes    = []
    global_lmax_hist= []
    torch.cuda.empty_cache()
    model = build_lora_model_silent(r_init)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    criterion = nn.CrossEntropyLoss()
    global_step = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            global_step += 1
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), 1.0)
            optimizer.step(); scheduler.step()
            running_loss += loss.item()
            correct += (out.logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
            # ── Per-layer CGRS check ─────────────────────────────────────────
            if global_step % K == 0:
                lmax_dict = compute_perlayer_fisher(model, probe_loader, NPROBE)
                gl_max = max(lmax_dict.values()) if lmax_dict else 0.0
                global_lmax_hist.append([global_step, gl_max])
                for i in range(12):
                    q_lmax = lmax_dict.get((i, "query"), 0.0)
                    v_lmax = lmax_dict.get((i, "value"), 0.0)
                    lay_lmax = max(q_lmax, v_lmax)
                    lambda_history[i].append([global_step, lay_lmax])
                    if cooldown_left[i] > 0:
                        cooldown_left[i] = max(0, cooldown_left[i] - K); continue
                    curr = layer_ranks[i]
                    if is_per_layer:
                        tau_i = max(tau_config.get(f"{i}_query", TAU_C2),
                                    tau_config.get(f"{i}_value", TAU_C2))
                    else:
                        tau_i = tau_config
                    r_idx = RANKLIST.index(curr) if curr in RANKLIST else 0
                    should_expand = (lay_lmax > tau_i) and (curr < P4["r_max"])
                    should_shrink = (not expand_only) and (lay_lmax < tau_i) and (curr > P4["r_min"])
                    new_r = curr
                    if should_expand and r_idx < len(RANKLIST) - 1:
                        new_r = RANKLIST[r_idx + 1]
                    elif should_shrink and r_idx > 0:
                        new_r = RANKLIST[r_idx - 1]
                    if new_r != curr:
                        cur_lr = float(scheduler.get_last_lr()[0])
                        model = transition_lora_rank(model, curr, new_r)
                        layer_ranks[i] = new_r
                        cooldown_left[i] = COOLDOWN
                        rank_history[i].append([global_step, new_r])
                        rank_changes.append({"step": global_step, "layer": i,
                                             "old_r": curr, "new_r": new_r, "lmax": lay_lmax})
                        remaining = max(total_steps - global_step, 1)
                        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                               lr=cur_lr, weight_decay=CONFIG["weight_decay"])
                        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining)
                        torch.cuda.empty_cache()
                        print(f"  Step {global_step:5d} Layer {i:2d}  {curr}→{new_r}  "
                              f"lmax={lay_lmax:.5f} τ={tau_i:.5f}")
        val_loss, val_acc = evaluate(model, val_loader)
        avg_r = np.mean(list(layer_ranks.values()))
        print(f"  Epoch {epoch}/{EPOCHS}  Val {val_acc:.2f}%  Avg Rank {avg_r:.1f}  "
              f"Changes so far: {len(rank_changes)}")
    test_loss, test_acc = evaluate(model, test_loader)
    final_ranks = dict(layer_ranks)
    avg_rank = float(np.mean(list(final_ranks.values())))
    lora_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  {run_name} DONE  Test Acc={test_acc:.2f}%  Avg Rank={avg_rank:.2f}  "
          f"LoRA Params={lora_params:,}  Changes={len(rank_changes)}")
    result = {
        "run_name": run_name, "r_init": r_init, "expand_only": expand_only,
        "test_acc": test_acc, "test_loss": test_loss, "avg_rank": avg_rank,
        "final_ranks": {str(k): v for k, v in final_ranks.items()},
        "total_lora_params": lora_params, "n_rank_changes": len(rank_changes),
        "rank_changes": rank_changes,
        "rank_history": {str(k): v for k, v in rank_history.items()},
        "lambda_history": {str(k): v for k, v in lambda_history.items()},
        "global_lmax_hist": global_lmax_hist,
    }
    rp = os.path.join(PHASE4_DIR, f"{run_name}_result.json")
    with open(rp, "w") as f: json.dump(result, f, indent=2)
    del model; torch.cuda.empty_cache()
    return result

print("run_perlayer_cgrs ready.")


run_perlayer_cgrs ready.


In [ ]:
# ----------------------------------------------------------------
# CELL 18 — Run Phase 4: C1, C2, C3, C3-half (resume-safe)
# ----------------------------------------------------------------
# Make sure Phase 2 was completed before running this cell.
runs_p4 = [
    ("PL_C1_tau75pct",  TAU_C1,      True),   # shared 75th pct
    ("PL_C2_tau50pct",  TAU_C2,      True),   # shared 50th pct
    ("PL_C3_PerLayer",  TAU_PER_LAYER, True),  # per-layer own λ_max (may invert correlation)
    ("PL_C3_Half",  TAU_C3_HALF,     True),   # corrected C3: 0.5 × λ_max
]
for run_name, tau_cfg, expand_only in runs_p4:
    if run_name in phase4_results and "test_acc" in phase4_results[run_name]:
        print(f"  {run_name} already done ({phase4_results[run_name]['test_acc']:.2f}%) — skipping.")
        continue
    result = run_perlayer_cgrs(run_name, tau_cfg, r_init=P4["r_init"], expand_only=expand_only)
    phase4_results[run_name] = result
    with open(p4_json_path, "w") as f: json.dump(phase4_results, f, indent=2)
    print(f"  Saved phase4_results.json")


  PL_C1_tau75pct already done (98.64%) — skipping.
  PL_C2_tau50pct
  tau=9.6259e-06  r_init=16  expand_only=True


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a dow

  Step   200 Layer  7  16→30  lmax=0.00002 τ=0.00001


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Step   200 Layer  8  16→30  lmax=0.00004 τ=0.00001


...

  Step  1000 Layer  8  52→64  lmax=0.00011 τ=0.00001


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Step  1000 Layer  9  52→64  lmax=0.00005 τ=0.00001


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Step  1000 Layer 11  30→52  lmax=0.00006 τ=0.00005


In [ ]:
# ----------------------------------------------------------------
# CELL 19 — Phase 4 Summary Table
# ----------------------------------------------------------------
with open(p4_json_path) as f: phase4_results = json.load(f)

print("=" * 80)
print("PHASE 4 — Per-Layer CGRS Results (CIFAR-10)")
print("=" * 80)
print(f"{'Run':30s}  {'r_init':6s}  {'Avg Rank':10s}  {'Test Acc':10s}  {'LoRA Params':12s}  {'Changes':8s}")
print("-" * 80)
for k, v in phase4_results.items():
    if "test_acc" not in v: continue
    print(f"{k:30s}  {v['r_init']:6d}  {v['avg_rank']:10.2f}  {v['test_acc']:8.2f}%  "
          f"{v['total_lora_params']:12,}  {v['n_rank_changes']:8d}")


## Final — Cross-Phase Comparison + Plots

In [ ]:
# ----------------------------------------------------------------
# CELL 20 — Full Cross-Phase Comparison Table (CIFAR-10)
# ----------------------------------------------------------------
with open(os.path.join(PHASE1_DIR, "results.json")) as f: results = json.load(f)
with open(p3_json_path) as f: phase3_results = json.load(f)
with open(p4_json_path) as f: phase4_results = json.load(f)

print("=" * 85)
print("COMPLETE COMPARISON — All Phases — CIFAR-10 / ViT-Base/16")
print("=" * 85)
print(f"{'Method':40s}  {'Type':12s}  {'Avg Rank':10s}  {'Test Acc':10s}  {'LoRA Params':12s}")
print("-" * 85)
# Phase 1 baselines
for method in ["Frozen backbone", "Full fine-tune"] + [f"LoRA r{r}" for r in LORA_RANKS]:
    if method not in results: continue
    v = results[method]; t = v.get("trainable_params", 0)
    r_str = method.replace("LoRA r", "")
    avg_r = int(r_str) if r_str.isdigit() else "-"
    print(f"{method:40s}  {'Static':12s}  {str(avg_r):>10s}  {v['test_acc']:8.2f}%  {t:12,}")
print("-" * 85)
# Phase 3 CGRS
for k, v in phase3_results.items():
    if "test_acc" not in v: continue
    print(f"{k:40s}  {'Dynamic/Global':12s}  {v['avg_rank']:10.2f}  {v['test_acc']:8.2f}%  {'—':>12s}")
print("-" * 85)
# Phase 4 per-layer
for k, v in phase4_results.items():
    if "test_acc" not in v: continue
    print(f"{k:40s}  {'Dynamic/PL':12s}  {v['avg_rank']:10.2f}  {v['test_acc']:8.2f}%  "
          f"{v['total_lora_params']:12,}")
print("=" * 85)


In [ ]:
# ----------------------------------------------------------------
# CELL 21 — Publication Plots
# ----------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150, "savefig.dpi": 300,
})

os.makedirs(PLOTS_DIR, exist_ok=True)
COLORS = {"fixed": "#8e9aaf", "global": "#457b9d", "perlayer": "#e63946",
          "fullft": "#2d6a4f", "frozen": "#b5b5b5"}

# ── Figure 1: λ_max & Trace vs Rank ──────────────────────────────────────────
with open(os.path.join(PHASE2_DIR, "curvature_results.json")) as f:
    curv = json.load(f)
valid_r = [r for r in LORA_RANKS if f"r{r}" in curv]
lmaxs   = [curv[f"r{r}"]["lambda_max"] for r in valid_r]
traces  = [curv[f"r{r}"]["trace"] for r in valid_r]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.semilogy(valid_r, lmaxs, "o-", color=COLORS["global"], linewidth=2, markersize=7)
ax1.set_xlabel("LoRA Rank r"); ax1.set_ylabel("λ_max (log scale)")
ax1.set_title("λ_max vs LoRA Rank — CIFAR-10 / ViT-Base/16", fontweight="bold")
ax1.grid(True, alpha=0.3)
ax2.semilogy(valid_r, traces, "s--", color=COLORS["perlayer"], linewidth=2, markersize=7)
ax2.set_xlabel("LoRA Rank r"); ax2.set_ylabel("Trace (log scale)")
ax2.set_title("Trace vs LoRA Rank — CIFAR-10 / ViT-Base/16", fontweight="bold")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
p = os.path.join(PLOTS_DIR, "fig1_curvature_vs_rank.png")
plt.savefig(p, bbox_inches="tight"); plt.close()
print(f"Saved: {p}")

# ── Figure 2: Accuracy vs LoRA Params Pareto frontier ────────────────────────
lora_accs   = [results.get(f"LoRA r{r}", {}).get("test_acc", None)  for r in valid_r]
lora_params = [results.get(f"LoRA r{r}", {}).get("trainable_params", None) for r in valid_r]
lora_accs   = [a for a in lora_accs   if a is not None]
lora_params = [p for p in lora_params if p is not None]
valid_r2    = [r for r in valid_r
               if results.get(f"LoRA r{r}", {}).get("test_acc") is not None]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([p/1e6 for p in lora_params], lora_accs,
        "o-", color=COLORS["fixed"], linewidth=2, markersize=8, label="Fixed-Rank LoRA")
# Phase 3 points
p3_accs = [(v["avg_rank"]*36864/1e6, v["test_acc"])
            for v in phase3_results.values() if "test_acc" in v]
if p3_accs:
    px, py = zip(*p3_accs)
    ax.scatter(list(px), list(py), c=COLORS["global"], marker="D", s=90, zorder=4,
               label="Global CGRS (Ph3)", edgecolors="white", linewidths=0.8)
# Phase 4 points
p4_colors = ["#e63946", "#f4a261", "#2a9d8f", "#8338ec"]
for idx, (k, v) in enumerate(phase4_results.items()):
    if "test_acc" not in v: continue
    ax.scatter(v["total_lora_params"]/1e6, v["test_acc"],
               c=p4_colors[idx % len(p4_colors)], marker="s", s=120, zorder=5,
               label=f"Per-Layer {k}", edgecolors="black", linewidths=0.8)
# Reference lines
if "Full fine-tune" in results:
    ax.axhline(results["Full fine-tune"]["test_acc"], color=COLORS["fullft"],
               linestyle=":", linewidth=1.8, alpha=0.8,
               label=f"Full FT {results['Full fine-tune']['test_acc']:.2f}%")
if "Frozen backbone" in results:
    ax.axhline(results["Frozen backbone"]["test_acc"], color=COLORS["frozen"],
               linestyle=":", linewidth=1.8, alpha=0.8,
               label=f"Frozen {results['Frozen backbone']['test_acc']:.2f}%")
ax.set_xlabel("LoRA Params (M)"); ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Accuracy vs. LoRA Params — CIFAR-10 / ViT-Base/16", fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
p = os.path.join(PLOTS_DIR, "fig2_pareto_frontier.png")
plt.savefig(p, bbox_inches="tight"); plt.close()
print(f"Saved: {p}")

# ── Figure 3: Per-layer final rank allocation ─────────────────────────────────
runs_to_plot = {k: v for k, v in phase4_results.items() if "final_ranks" in v}
if runs_to_plot:
    n_runs = len(runs_to_plot); x = np.arange(12); width = 0.8 / max(n_runs, 1)
    bar_colors = ["#e63946", "#f4a261", "#2a9d8f", "#8338ec"]
    fig, ax = plt.subplots(figsize=(14, 5))
    for idx, (k, v) in enumerate(runs_to_plot.items()):
        fr = [v["final_ranks"].get(str(i), P4["r_init"]) for i in range(12)]
        offset = (idx - n_runs / 2 + 0.5) * width
        ax.bar(x + offset, fr, width*0.9, color=bar_colors[idx % len(bar_colors)],
               alpha=0.85, label=f"{k} (Acc={v['test_acc']:.2f}%)", edgecolor="white")
    ax.axhline(P4["r_init"], color="gray", linestyle="--", linewidth=1.2, alpha=0.6,
               label=f"r_init={P4['r_init']}")
    ax.set_xticks(x); ax.set_xticklabels([f"L{i}" for i in range(12)])
    ax.set_xlabel("Transformer Layer"); ax.set_ylabel("Final LoRA Rank")
    ax.set_title("Per-Layer Final Rank — Per-Layer CGRS (CIFAR-10)", fontweight="bold")
    ax.legend(fontsize=8.5); ax.grid(True, alpha=0.3, axis="y")
    ax.set_ylim(0, P4["r_max"] + 8)
    plt.tight_layout()
    p = os.path.join(PLOTS_DIR, "fig3_perlayer_rank_allocation.png")
    plt.savefig(p, bbox_inches="tight"); plt.close()
    print(f"Saved: {p}")

print("\nAll plots saved to:", PLOTS_DIR)


In [ ]:
# ----------------------------------------------------------------
# CELL 22 — Correlation Analysis (Curvature vs Final Rank — Phase 4)
# ----------------------------------------------------------------
from scipy.stats import pearsonr, spearmanr

if "r16" in curvature_results:
    pl16 = curvature_results["r16"]["per_layer"]
    print("\nCorrelation: Phase 2 λ_max vs Final LoRA Rank (Per-Layer CGRS)\n")
    print(f"{'Run':25s}  {'Pearson r':12s}  {'Spearman ρ':12s}  {'Spearman p':12s}")
    print("-" * 65)
    for run_name, v in phase4_results.items():
        if "final_ranks" not in v: continue
        lmax_vals  = [max(pl16.get(f"{i}_query", 0.0), pl16.get(f"{i}_value", 0.0))
                      for i in range(12)]
        rank_vals  = [v["final_ranks"].get(str(i), P4["r_init"]) for i in range(12)]
        if len(set(rank_vals)) < 2:
            print(f"{run_name:25s}  (all same rank — no variance)")
            continue
        pr, pp = pearsonr(lmax_vals, rank_vals)
        sr, sp = spearmanr(lmax_vals, rank_vals)
        print(f"{run_name:25s}  {pr:12.4f}  {sr:12.4f}  {sp:12.4e}")
    print("\nNote: Positive correlation = high curvature → high rank (expected direction).")
    print("      Negative correlation (C3) = C3 anomaly — investigate.")


In [ ]:
# ----------------------------------------------------------------
# CELL 23 — File Verification + Zip Command
# ----------------------------------------------------------------
print("\nFile Verification:")
for d, label in [(PHASE1_DIR, "Phase 1"), (PHASE2_DIR, "Phase 2"),
                 (PHASE3_DIR, "Phase 3"), (PHASE4_DIR, "Phase 4"), (PLOTS_DIR, "Plots")]:
    files = os.listdir(d) if os.path.exists(d) else []
    print(f"  {label}: {len(files)} files in {d}")

print("\nTo zip all results:")
print(f"  zip -r CGRS_CIFAR10_results.zip {PROJECT_BASE}")
